# 01 · Explore WHO competency domains

Loads `data/who_strategy/competencies.csv`, inspects the domain descriptions, embeds them, and looks at how similar the domains are to each other (a sanity check that the domains are reasonably distinct).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root on path

import numpy as np
import pandas as pd
from src import competencies
from src.models import load_embedding_model
from src import alignment

comp_df = competencies.load_competencies()
comp_df

In [ ]:
# Embed the domain descriptions (falls back to TF-IDF if sentence-transformers is absent)
model = load_embedding_model()
texts = competencies.embedding_texts(comp_df)
model.fit(texts)
emb = model.encode(texts)
print('backend:', model.name, '| embeddings:', emb.shape)

In [ ]:
# Domain-to-domain cosine similarity: off-diagonal values should be well below 1.
import seaborn as sns, matplotlib.pyplot as plt
sim = alignment.compute_similarity(emb, emb)
ax = sns.heatmap(sim, xticklabels=comp_df.competency_id, yticklabels=comp_df.competency_id,
                 cmap='mako', annot=True, fmt='.2f')
ax.set_title('WHO competency domain similarity (distinctness check)')
plt.show()